# CircleID Baseline

This notebook provides a minimal ResNet18 baseline for the CircleID Kaggle challenges.

It supports two tracks:
- **Writer Identification** (with an "unknown writer" option via a simple confidence threshold)
- **Pen Classification**

## What it does
1. Loads `train.csv` and `test.csv`
2. Creates a validation split from `train.csv`
3. Fine-tunes an ImageNet-pretrained ResNet18
4. Writes a Kaggle submission depending on the mode: `submission_writer.csv` or `submission_pen.csv`

## Input format
- `train.csv`: `image_id`, `image_path`, `writer_id`, `pen_id`
- `test.csv`: `image_id`, `image_path`
- The `dataset` folder contains the images as PNG files.

`writer_id` is a string in the form `W01`, `W02`, ..., `W51`.

`pen_id` is an integer in the range 1 - 8.

## Notes
- This notebook is designed to be easily runable and tunable (for example augmentations, threshold, architecture).
- The unknown writer handling is a simple baseline heuristic and a good place to improve.

## References
- ResNet: He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. In Proceedings of the IEEE conference on computer vision and pattern recognition (pp. 770-778).
- ImageNet: Deng, J., Dong, W., Socher, R., Li, L. J., Li, K., & Fei-Fei, L. (2009, June). Imagenet: A large-scale hierarchical image database. In 2009 IEEE conference on computer vision and pattern recognition (pp. 248-255). Ieee.


## Libraries

Install third-party libraries. This step can be skipped on Kaggle, as most dependencies are preinstalled.

In [ ]:
!pip install -q numpy pandas pillow torch torchvision tqdm

## Configuration

Select the task, set paths, and adjust a hyperparameters.

Notes:
- On Kaggle the input data is located under `/kaggle/input/...` (mounted as read-only). Outputs should be written to `/kaggle/working/`.
- `DATASET_DIR` should contain `train.csv` and `test.csv`.
- If running locally, replace `DATASET_DIR` and `OUTPUT_DIR` with your own paths.

In [ ]:
TASK = "writer"  # "writer" or "pen"

# When running in Kaggle notebooks:
# Inputs are read-only under /kaggle/input/
# Output must be written to /kaggle/working/
DATASET_DIR = f"/kaggle/input/icdar-2026-circle-id-{TASK}-identification/"   # folder containing train.csv and test.csv
OUTPUT_DIR = "/kaggle/working"

# Local alternative
# DATASET_DIR = "YOUR_DATASET_PATH"
# OUTPUT_DIR = "YOUR_OUTPUT_PATH"

# Training hyperparameters
EPOCHS = 10
BATCH_SIZE = 128
LEARNING_RATE = 3e-4
IMG_SIZE = 224
SEED = 0

# Holdout validation fraction (set 0.0 to disable)
VAL_FRAC = 0.2

# Writer task only: Below this confidence, writers are predicted as unknown (-1)
WRITER_UNKNOWN_THRESHOLD = 0.9

# Output path
CKPT_PATH      = f"{OUTPUT_DIR}/baseline_{TASK}.pt"
BEST_CKPT_PATH = f"{OUTPUT_DIR}/baseline_{TASK}_best.pt"
LOG_PATH       = f"{OUTPUT_DIR}/log_{TASK}.json"


## Imports

Use PyTorch and torchvision to finetune for the pretrained ResNet18 baseline.

In [ ]:
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from tqdm import tqdm

## Utilities

Define helper functions used in this notebook.

In [ ]:
def set_seeds(seed: int):
    """Make training reproducible"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def generate_label_maps(train_df: pd.DataFrame):
    """Build label - index maps from the training data."""
    if TASK == "writer":
        labels = sorted(train_df["writer_id"].astype(str).unique().tolist())
    elif TASK == "pen":
        labels = sorted(train_df["pen_id"].astype(str).unique().tolist())
    else:
        raise ValueError("TASK must be 'writer' or 'pen'")

    label_map = {label: i for i, label in enumerate(labels)}
    index_map = {i: label for label, i in label_map.items()}

    return label_map, index_map

def random_split(df: pd.DataFrame, val_frac: float, seed: int):
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    
    val_size = int(round(val_frac * len(df)))

    val_df = df.iloc[:val_size].copy()
    train_df = df.iloc[val_size:].copy()

    return train_df, val_df

## Dataset

Load images using paths from train.csv and test.csv.

In [ ]:
class CircleDataset(Dataset):
    """
    Dataset for circles.
    - df: Pandas DataFrame
    - return_label: Return labels as a tuple (x, y) otherweise return (x, image_id)
    - augment: Apply augmentation (should only be enabled for training)
    """
    def __init__(self, df: pd.DataFrame, img_root: Path, return_label: bool, augment: bool):
        self.df = df.reset_index(drop=True)
        self.img_root = img_root
        self.return_label = return_label

        mean = ResNet18_Weights.DEFAULT.transforms().mean
        std = ResNet18_Weights.DEFAULT.transforms().std

        if augment:
            self.transforms = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.RandomRotation(degrees=10),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])
        else:
            self.transforms = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i: int):
        row = self.df.iloc[i]
        image_id = str(row["image_id"])
        rel_path = str(row["image_path"])
        img_path = self.img_root / rel_path

        img = Image.open(img_path).convert("RGB")
        x = self.transforms(img)

        if self.return_label:
            y = int(row["y"])
            return x, y
        else:
            return x, image_id

## Model

This section defines functions to create the ResNet18 model and the classification head. It also includes functions for the training and validation process.

In [ ]:
def build_model(num_classes: int) -> nn.Module:
    """ResNet18 backbone and a linear classifier head."""
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def train_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    i = 0

    for x, y in tqdm(loader, desc="Train", unit="batch"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = F.cross_entropy(logits, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        total += loss.item() * batch_size
        i += batch_size

    return total / i

@torch.no_grad()
def evaluate(model, loader, device):
    """Compute loss + top-1 accuracy on validation set."""
    model.eval()
    total_loss = 0.0
    correct = 0
    i = 0

    for x, y in tqdm(loader, desc="Val", unit="batch"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = F.cross_entropy(logits, y)

        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        i += batch_size

    return total_loss / i, correct / i

@torch.no_grad()
def predict(model, loader, device, idx_map):
    """Predict for test.csv and format rows for Kaggle submission."""
    model.eval()
    out = []

    for x, image_ids in tqdm(loader, desc="Predict", unit="batch"):
        x = x.to(device, non_blocking=True)
        logits = model(x)

        if TASK == "writer":
            # Simple unknown-writer heuristic: low softmax confidence => -1
            probs = F.softmax(logits, dim=1)
            pred_confs, pred_indices = probs.max(dim=1)
            for img_id, pred_conf, pred_idx in zip(image_ids, pred_confs.cpu().numpy(), pred_indices.cpu().numpy()):
                if float(pred_conf) < WRITER_UNKNOWN_THRESHOLD:
                    out.append((img_id, "-1"))
                else:
                    out.append((img_id, idx_map[int(pred_idx)]))
        elif TASK == "pen":
            pred_indices = logits.argmax(dim=1).cpu().numpy()
            for img_id, pi in zip(image_ids, pred_indices):
                out.append((img_id, int(idx_map[int(pi)])))
        else:
            raise ValueError("TASK must be 'writer' or 'pen'")

    return out

# Training and create submission

This code loads the dataset, creates a validation split, trains the model, saves the best checkpoint, and writes the Kaggle submission file for the selected task.

In [ ]:
set_seeds(SEED)

dataset_dir = Path(DATASET_DIR)

train_df = pd.read_csv(dataset_dir / "train.csv")
test_df  = pd.read_csv(dataset_dir / "test.csv")

label_map, idx_map = generate_label_maps(train_df)

# Add target column "y" for the chosen task
if TASK == "writer":
    train_df["y"] = train_df["writer_id"].astype(str).map(label_map).astype(int)
else:
    train_df["y"] = train_df["pen_id"].astype(str).map(label_map).astype(int)

# Create validation set
train_df, val_df = random_split(train_df, val_frac=VAL_FRAC, seed=SEED)
print(f"Train samples: {len(train_df)} | Validation samples: {len(val_df)} ({VAL_FRAC:.2f}%)")
if TASK == "writer":
    print("Note: Validation accuracy is calculated only on known writers.")

# For reproducibility
log = {
    "task": TASK,
    "seed": SEED,
    "label_map": label_map,
    "idx_map": idx_map,
    "writer_unknown_threshold": WRITER_UNKNOWN_THRESHOLD,
    "val_frac": VAL_FRAC,
}
Path(LOG_PATH).write_text(json.dumps(log, indent=4), encoding="utf-8")
print(f"Saved log to {LOG_PATH}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_model(num_classes=len(label_map)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Train dataset: labels + augmentation
train_ds = CircleDataset(train_df, img_root=dataset_dir, return_label=True, augment=True)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

# Val dataset: labels and no augmentation
val_ds = CircleDataset(val_df, img_root=dataset_dir, return_label=True, augment=False)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

best_acc = -1.0
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, device)

    val_loss, val_acc = evaluate(model, val_loader, device)
    print(f"[Epoch {epoch + 1}/{EPOCHS}] Train loss: {train_loss:.4f} | val loss: {val_loss:.4f} | val acc: {val_acc:.4f}")

    # Save best model by validation accuracy
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({"model": model.state_dict()}, BEST_CKPT_PATH)

# Save last checkpoint
torch.save({"model": model.state_dict()}, CKPT_PATH)
print(f"Saved last checkpoint: {CKPT_PATH}")
print(f"Saved best checkpoint: {BEST_CKPT_PATH} (best val acc={best_acc:.4f})")

# Load best checkpoint for test prediction
model_state = torch.load(BEST_CKPT_PATH, map_location=device)
model.load_state_dict(model_state["model"])

# Predict labels for test set
test_ds = CircleDataset(test_df, img_root=dataset_dir, return_label=False, augment=False)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

predictions = predict(model, test_loader, device, idx_map)

if TASK == "writer":
    sub = pd.DataFrame(predictions, columns=["image_id", "writer_id"])
    out_name = "submission_writer.csv"
else:
    sub = pd.DataFrame(predictions, columns=["image_id", "pen_id"])
    out_name = "submission_pen.csv"

sub.to_csv(out_name, index=False)
print(f"Wrote: {out_name}")